# Periapical Lesion Training (Cloud GPU)
Train all 3 models on full dataset using T4 GPU.

**How to use:**
1. Upload project to `/content/drive/MyDrive/Periapical Lesions/`
2. Set `mode = 'colab'` below
3. Run all cells
4. Models are saved to `Periapical Lesions/models/` on Drive

In [ ]:
# ============================================================
# MODE TOGGLE
# ============================================================
mode = 'colab'   # 'local' = tiny test (50 samples, 3 epochs)
                 # 'colab' = full training on T4 GPU

print(f"Mode: {mode}")

## 1. Mount Google Drive & Setup Project

In [ ]:
import sys, os, json, warnings, shutil, subprocess
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'streamlit',
                'opencv-python', 'tqdm', 'scikit-learn', 'pyyaml'], check=True)
subprocess.run(['pip', 'install', '-q', 'torchvision', '--upgrade'], check=True)

## 2. Override Config for Full Training

In [ ]:
from src.config import *
import random
random.seed(RANDOM_SEED)

# Override for full training on T4
FULL_TRAIN = True
NUM_SAMPLES = None
EPOCHS_CLS = 30
EPOCHS_DET = 150
EPOCHS_SEG = 100
DEVICE = DEVICE_OVERRIDE
BATCH_SIZE = 32
print(f'Full: {EPOCHS_CLS} cls / {EPOCHS_DET} det / {EPOCHS_SEG} seg epochs')
print(f'Device: {DEVICE}, Batch: {BATCH_SIZE}')

os.makedirs(MODELS_DIR, exist_ok=True)

## 3. Data Preparation

In [ ]:
from src.data_utils import *

print('Scanning dataset...')
original_dir, aug_dir, annot_dir = scan_dataset()
print(f'  Original: {original_dir}')
print(f'  Annot:    {annot_dir}')

print('Building manifest...')
manifest = build_dataset_manifest(original_dir, annot_dir, sample_size=NUM_SAMPLES)
print(f'  Samples: {len(manifest)}')
dist = get_class_distribution(manifest)
print(f'  Distribution: {dict(dist)}')

print('Splitting...')
splits = split_dataset(manifest)
for k, v in splits.items():
    print(f'  {k}: {len(v)}')

print('Preparing data...')
cls_records = prepare_classification_data(splits)
det_records, yaml_path = prepare_detection_data(splits)
seg_records = prepare_segmentation_data(splits)
print(f'  Classification: {len(cls_records)}')
print(f'  Detection:      {len(det_records)}')
print(f'  Segmentation:   {len(seg_records)}')

## 4. Train Classification

In [ ]:
import shutil
shutil.rmtree('/content/code/src/__pycache__', ignore_errors=True)

import torch, os, json
from src.train_classification import train_classification, evaluate_classification
from src.models import get_classifier

train_rec = [r for r in cls_records if r['split'] == 'train']
val_rec = [r for r in cls_records if r['split'] == 'val']
test_rec = [r for r in cls_records if r['split'] == 'test']
print(f'Train: {len(train_rec)}, Val: {len(val_rec)}, Test: {len(test_rec)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('\nTraining classifier...')
cls_model, cls_history = train_classification(train_rec, val_rec, num_epochs=30, batch_size=32, device=device)

print('\nEvaluating...')
cls_metrics, _, _, _ = evaluate_classification(cls_model, test_rec, device=device)
print(f'  Accuracy: {cls_metrics["accuracy"]:.4f}')
print(f'  F1-Score: {cls_metrics["f1_score"]:.4f}')

cls_out = os.path.join('/content/code/models', 'classifier.pth')
torch.save(cls_model.state_dict(), cls_out)
print(f'Saved: {cls_out}')

## 5. Train Detection

In [ ]:
import torch, os, json, shutil
from src.train_detection import train_detection

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Training detector...')
det_model, det_metrics = train_detection(yaml_path, epochs=150, batch_size=16, device=device)
print(f'  mAP50: {det_metrics["mAP50"]:.4f}')
print(f'  Recall: {det_metrics["recall"]:.4f}')

det_src = os.path.join('/content/code/outputs', 'detection', 'yolo_run', 'weights', 'best.pt')
det_dst = os.path.join('/content/code/models', 'best.pt')
if os.path.exists(det_src):
    shutil.copy2(det_src, det_dst)
    print(f'Saved: {det_dst}')

with open(os.path.join('/content/code/outputs', 'detection', 'metrics.json'), 'w') as f:
    json.dump(det_metrics, f, indent=2)

## 6. Train Segmentation

In [ ]:
import torch, os
from src.train_segmentation import train_segmentation, evaluate_segmentation

seg_train = [r for r in seg_records if r['split'] == 'train']
seg_val = [r for r in seg_records if r['split'] == 'val']
seg_test = [r for r in seg_records if r['split'] == 'test']
print(f'Train: {len(seg_train)}, Val: {len(seg_val)}, Test: {len(seg_test)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('\nTraining segmenter...')
seg_model, seg_history = train_segmentation(seg_train, seg_val, num_epochs=100, batch_size=32, device=device)

print('\nEvaluating...')
seg_metrics, _, _ = evaluate_segmentation(seg_model, seg_test, device=device)
print(f'  Dice: {seg_metrics["dice"]:.4f}')
print(f'  IoU: {seg_metrics["iou"]:.4f}')

seg_out = os.path.join('/content/code/models', 'segmenter.pth')
torch.save(seg_model.state_dict(), seg_out)
print(f'Saved: {seg_out}')

## 7. Copy models back to Drive

In [ ]:
# Copy models back to Drive
drive_models = '/content/drive/MyDrive/Periapical Lesions/models'
os.makedirs(drive_models, exist_ok=True)
for fname in ['classifier.pth', 'best.pt', 'segmenter.pth']:
    src = os.path.join('/content/code/models', fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_models, fname))
        print(f'Copied {fname} to Drive')

print('\nDone!')